# Library

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# Load Clean Dataset

In [81]:
df = pd.read_parquet(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/processed dataset/final_df_(distance).parquet"
)

final_features = [
    "total_price",
    "total_freight_value",
    "total_product_weight_g",
    "average_product_length_cm",
    "average_product_height_cm",
    "item_count",
    "max_payment_installments",
    "average_distance_km"
]

experiment_columns = [
    "order_id",
    *final_features,
    "is_delayed"
]

model_df = df[experiment_columns].dropna().copy()

print(model_df.shape)
print(model_df.isna().sum().sum())

(95960, 10)
0


In [82]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    model_df,
    test_size=0.2,
    random_state=42,
    stratify=model_df["is_delayed"]
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (76768, 10)
Test: (19192, 10)


In [83]:
train_idx = train_df.index
test_idx = test_df.index

print("Train indices:", len(train_idx))
print("Test indices:", len(test_idx))

Train indices: 76768
Test indices: 19192


# Load Missing Dataset

In [84]:
from pathlib import Path

data_folder = Path("/Users/sivhoung/Desktop/Year 4 intern project/Project code /dataset/exp_data")

injection_data = {}

for file in data_folder.glob("*.parquet"):
    injection_data[file.stem] = pd.read_parquet(file)

print(injection_data.keys())

dict_keys(['mar_50', 'mnar_50', 'mcar_30', 'mar_30', 'mnar_30', 'mcar_50', 'mcar_10', 'mar_10', 'mnar_10'])


### 修复说明（2026-09-29）

从「Model function」这格开始往下的内容整个重写，改了这几处：

1. **Mean/MICE 结果重复的 bug**：原来记录 Mean 结果时用的是上一个 cell（MICE）残留的 `y_pred`/`y_prob`。现在用 `run_model_and_record()`，训练/预测/记录在同一次调用里完成，不依赖任何跨 cell 变量。
2. **`rf` 未定义会直接报错**：原 Median 那格写 `xgb_model = XGBClassifier(...)` 却调用 `rf.fit()`，`rf` 在那之前根本没定义过。Restart & Run All 会 `NameError`。
3. **模型统一**：四种方法统一用 `RandomForestClassifier(class_weight="balanced")`（XGBoost 没设 `scale_pos_weight`，两份 notebook 里都只预测多数类，Precision=Recall=0）。
4. **对齐方式**：原来 `mcar_10.index = model_df.index` 是硬套 index。现在用 `order_id` 显式过滤 train/test，不依赖行顺序假设。
5. **原来只做了 MCAR 10% 一个场景**：现在循环跑完全部 9 个（MCAR/MAR/MNAR × 10/30/50%）× 4 种方法 = 36 行，另外加 2 个 baseline（Full data / Drop column）。

Restart & Run All 之后，最后 `downstream_results_df` 应该是 38 行。


## 目标列 + train/test 的 order_id 集合

In [ ]:
target_col = "total_freight_value"

train_order_ids = set(train_df["order_id"])
test_order_ids = set(test_df["order_id"])


def split_by_order_id(injected_df):
    inj_train = injected_df[injected_df["order_id"].isin(train_order_ids)].copy()
    inj_test = injected_df[injected_df["order_id"].isin(test_order_ids)].copy()
    return inj_train, inj_test


## 统一的填补函数（train-only 统计量，test 只 transform，不泄漏）

In [ ]:
def impute_mean(train_df, test_df, target):
    train_out, test_out = train_df.copy(), test_df.copy()
    value = train_out[target].mean()
    train_out[target] = train_out[target].fillna(value)
    test_out[target] = test_out[target].fillna(value)
    return train_out, test_out


def impute_median(train_df, test_df, target):
    train_out, test_out = train_df.copy(), test_df.copy()
    value = train_out[target].median()
    train_out[target] = train_out[target].fillna(value)
    test_out[target] = test_out[target].fillna(value)
    return train_out, test_out


def impute_knn(train_df, test_df, features, n_neighbors=5):
    train_out, test_out = train_df.copy(), test_df.copy()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_out[features])
    X_test_scaled = scaler.transform(test_out[features])

    imputer = KNNImputer(n_neighbors=n_neighbors, weights="uniform")
    X_train_imp_scaled = imputer.fit_transform(X_train_scaled)
    X_test_imp_scaled = imputer.transform(X_test_scaled)

    train_out[features] = scaler.inverse_transform(X_train_imp_scaled)
    test_out[features] = scaler.inverse_transform(X_test_imp_scaled)
    return train_out, test_out


def impute_mice(train_df, test_df, features, n_iter=10, random_seed=42):
    train_out, test_out = train_df.copy(), test_df.copy()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(train_out[features])
    X_test_scaled = scaler.transform(test_out[features])

    imputer = IterativeImputer(max_iter=n_iter, random_state=random_seed)
    X_train_imp_scaled = imputer.fit_transform(X_train_scaled)
    X_test_imp_scaled = imputer.transform(X_test_scaled)

    train_out[features] = scaler.inverse_transform(X_train_imp_scaled)
    test_out[features] = scaler.inverse_transform(X_test_imp_scaled)
    return train_out, test_out


## 统一的模型 + 记录函数（原子操作）

In [ ]:
def run_model_and_record(results, train_df, test_df, mechanism, missing_rate, method,
                          features=final_features):
    X_train = train_df[features]
    y_train = train_df["is_delayed"]
    X_test = test_df[features]
    y_test = test_df["is_delayed"]

    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "mechanism": mechanism,
        "missing_rate": missing_rate,
        "method": method,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob)
    })


## 主循环：9 个注入数据集 × 4 种填补方法 = 36 行

In [ ]:
downstream_results = []

for name, injected_df in injection_data.items():
    mechanism, rate_str = name.split("_")
    mechanism = mechanism.upper()
    rate = int(rate_str) / 100

    inj_train, inj_test = split_by_order_id(injected_df)

    m_train, m_test = impute_mean(inj_train, inj_test, target_col)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "Mean")

    m_train, m_test = impute_median(inj_train, inj_test, target_col)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "Median")

    m_train, m_test = impute_knn(inj_train, inj_test, final_features)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "KNN")

    m_train, m_test = impute_mice(inj_train, inj_test, final_features)
    run_model_and_record(downstream_results, m_train, m_test, mechanism, rate, "MICE")

downstream_results_df = pd.DataFrame(downstream_results)
downstream_results_df = downstream_results_df.drop_duplicates().sort_values(
    ["mechanism", "missing_rate", "method"]
).reset_index(drop=True)

print("Rows:", len(downstream_results_df))
downstream_results_df


## Baseline 1：Full data（完全没有缺失值注入）

In [ ]:
run_model_and_record(
    downstream_results,
    train_df, test_df,
    mechanism="Baseline", missing_rate=0.0, method="Full_data"
)


## Baseline 2：Drop column（训练时直接不用 total_freight_value）

In [ ]:
features_no_target = [f for f in final_features if f != target_col]

run_model_and_record(
    downstream_results,
    train_df, test_df,
    mechanism="Baseline", missing_rate=0.0, method="Drop_column",
    features=features_no_target
)

downstream_results_df = pd.DataFrame(downstream_results)
downstream_results_df = downstream_results_df.drop_duplicates().sort_values(
    ["mechanism", "missing_rate", "method"]
).reset_index(drop=True)

print("Total rows:", len(downstream_results_df))
downstream_results_df


## 保存结果

In [ ]:
downstream_results_df.to_csv(
    "/Users/sivhoung/Desktop/Year 4 intern project/Project code /results/downstream_results.csv",
    index=False
)
